<h1>売上データ分析例</h1>

模擬講義で使用するための仮想的なレストラン売り上げデータの分析例です．あえて途中で止めています．

ライブラリ読み込み

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
import pandas as pd
import sklearn
import scipy
import pandas as pd
import seaborn as sns
import statsmodels.api as sm
import statsmodels.formula.api as smf
import matplotlib.font_manager as fm
from pathlib import Path

表形式のデータを読み込み

In [ ]:
sales=pd.read_csv("../data/salesmissingdata.csv")   # データ読み込み
sales.index=range(1,32)                          # 行名を日付にそろえる

読み込んだ表形式の中身を確認

In [ ]:
sales

データ抽出の例：26日以降のデータのみ抽出

In [ ]:
sales.query('日付>=26')

売上データの無い日・ある日を抽出

In [ ]:
sales.query('売上.isnull()')

In [ ]:
sales.query('~売上.isnull()')   #~は否定を表す

売上データのある日を使い統計モデルを作る．そのためのデータセットを train とする．

In [ ]:
train=sales.query('~売上.isnull()') 

In [ ]:
train

売上データの無い日を予測したい．これを test とする．

In [ ]:
test=sales.query('売上.isnull()')

In [ ]:
test

日本語を図に入れるための前処理

In [ ]:
# アップロードしたフォントファイルのパス
font_path = Path("../fonts/NotoSansJP-Regular.ttf")
# Matplotlib にフォントを登録
fm.fontManager.addfont(str(font_path))
# フォント名を取得して rcParams に設定
font_name = fm.FontProperties(fname=str(font_path)).get_name()
plt.rcParams["font.family"] = font_name
# マイナス記号の文字化け対策
plt.rcParams["axes.unicode_minus"] = False

気温と売上の散布図を描いてみる

In [ ]:
plt.scatter(sales['気温'],sales['売上'])
plt.xlabel('気温')
plt.ylabel('売上')
plt.title('気温・売上散布図')
plt.show()

気温と売上の相関係数を見てみる

In [ ]:
sales[['気温','売上']].corr()

曜日別/降雨有無別の売上の箱ひげ図を描いてみる

In [ ]:
sns.boxplot(x='曜日',y='売上',data=sales)
plt.show()

In [ ]:
sns.boxplot(x='降雨',y='売上',data=sales)
plt.show()

回帰分析：単回帰，気温と売上

In [ ]:
result = smf.ols( formula='売上 ~ 気温', data= train).fit()
result.summary()

In [ ]:
a=result.params.気温               # 求められた傾き
b=result.params.Intercept          # 求められた切片
xmin=np.min(sales['気温'])-1       # 気温の最小値よりも1小さい値をxmin（直線の描画範囲の最小x座標)
xmax=np.max(sales['気温'])+1       # 気温の最大値よりも1大きい値をxmax（直線の描画範囲の最大x座標)
ymin=a*xmin+b                      # 直線の式に代入して端点設定
ymax=a*xmax+b

plt.scatter(sales['気温'],sales['売上'])
plt.plot([xmin,xmax],[ymin,ymax],color='red')     # 端点同士を結ぶ直線で回帰直線プロット
plt.xlabel('気温')
plt.ylabel('売上')
plt.title('気温・売上の回帰')
plt.show()

売上データのない３日分の売上を気温から予測

In [ ]:
result.predict(test)

「実際のデータ」の読み込み

In [ ]:
fulldata=pd.read_csv("../data/fulldata.csv")
fulldata.index=range(1,32)

「実際のデータ」の３日分の表に，予測した３日分を合成した表を作成(predicted_dataset)

In [ ]:
predicted_dataset=fulldata.filter(items=range(29,32),axis=0)
predicted_dataset['予測']=result.predict(test)

In [ ]:
predicted_dataset

予測と売上の差

In [ ]:
predicted_dataset['予測']-predicted_dataset['売上']

「二乗平均平方根誤差」という量により，３日間の平均的な誤差を計算

In [ ]:
sklearn.metrics.root_mean_squared_error(predicted_dataset['売上'],predicted_dataset['予測'])

重回帰：今度は気温だけでなく降雨の効果を考える．

In [ ]:
result2 = smf.ols( formula='売上 ~ 気温 + 降雨', data= sales).fit()
result2.summary()

In [ ]:
result2.predict(test)

In [ ]:
predicted_dataset2=fulldata.filter(items=range(29,32),axis=0)
predicted_dataset2['予測']=result2.predict(test)

In [ ]:
predicted_dataset2['予測']-predicted_dataset2['売上']

In [ ]:
sklearn.metrics.root_mean_squared_error(predicted_dataset2['売上'],predicted_dataset2['予測'])

他の効果も考えてみよう．